In [1]:
import os
import numpy as np
from PIL import Image

In [2]:
X = []
y = []

In [3]:
normal_folder = "../datasets/train/NORMAL"

for filename in os.listdir(normal_folder):

    image_path = os.path.join(normal_folder, filename)

    img = Image.open(image_path)

    img = img.convert("L")
    img = img.resize((64,64))

    img_array = np.array(img)

    flat_img = img_array.flatten()

    X.append(flat_img)

    y.append(0)

print("Normal Images Loaded")

Normal Images Loaded


In [4]:
pneumonia_folder = "../datasets/train/PNEUMONIA"

for filename in os.listdir(pneumonia_folder):

    image_path = os.path.join(pneumonia_folder, filename)

    img = Image.open(image_path)

    img = img.convert("L")
    img = img.resize((64,64))

    img_array = np.array(img)

    flat_img = img_array.flatten()

    X.append(flat_img)

    y.append(1)

print("Pneumonia Images Loaded")

Pneumonia Images Loaded


In [5]:
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(5216, 4096)
(5216,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(4172, 4096)
(1044, 4096)


In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Training Complete!")

Training Complete!


In [8]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9549808429118773


In [11]:
import os

os.makedirs("../models", exist_ok=True)
import joblib

joblib.dump(model, "../models/pneumonia_model.pkl")

print("Model Saved!")

Model Saved!


In [12]:
from PIL import Image
import numpy as np

def predict_xray(image_path):

    img = Image.open(image_path)

    img = img.convert("L")
    img = img.resize((64,64))

    img_array = np.array(img)

    flat_img = img_array.flatten().reshape(1,-1)

    prediction = model.predict(flat_img)

    if prediction[0] == 0:
        print("Prediction: NORMAL")
    else:
        print("Prediction: PNEUMONIA")

In [14]:
import os

print(os.path.exists("../models/pneumonia_model.pkl"))

True


In [15]:
predict_xray("../datasets/test/NORMAL/IM-0007-0001.jpeg")

Prediction: PNEUMONIA


In [16]:
predict_xray("../datasets/test/PNEUMONIA/person100_bacteria_475.jpeg")

Prediction: PNEUMONIA


In [17]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))

[[260  27]
 [ 20 737]]


In [18]:
predict_xray("../datasets/test/NORMAL/IM-0007-0001.jpeg")

Prediction: PNEUMONIA


In [19]:
predict_xray("../datasets/test/PNEUMONIA/person100_bacteria_475.jpeg")

Prediction: PNEUMONIA


In [20]:
normal_files = [
    "IM-0001-0001.jpeg",
    "IM-0003-0001.jpeg",
    "IM-0005-0001.jpeg",
    "IM-0006-0001.jpeg",
    "IM-0007-0001.jpeg",
]

In [23]:
print("Start")

from PIL import Image
import numpy as np

img = Image.open("../datasets/test/NORMAL/IM-0007-0001.jpeg")

print("Image Loaded")

img = img.convert("L")
img = img.resize((64,64))

print("Image Processed")

test_vector = np.array(img).flatten()

print("Vector Created")
print("Shape:", test_vector.shape)

pred = model.predict(test_vector.reshape(1,-1))[0]

print("Prediction:", pred)

print("End")

Start
Image Loaded
Image Processed
Vector Created
Shape: (4096,)
Prediction: 1
End


In [28]:
import os
from PIL import Image
import numpy as np

normal_folder = "../datasets/test/NORMAL"

normal_count = 0
pneumonia_count = 0

for file in os.listdir(normal_folder)[:30]:

    img = Image.open(os.path.join(normal_folder, file))
    img = img.convert("L")
    img = img.resize((64,64))

    x = np.array(img).flatten().reshape(1,-1)

    pred = model.predict(x)[0]

    if pred == 0:
        normal_count += 1
    else:
        pneumonia_count += 1

print("Predicted NORMAL:", normal_count)
print("Predicted PNEUMONIA:", pneumonia_count)

Predicted NORMAL: 4
Predicted PNEUMONIA: 26
